### Graph v10: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [1]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [2]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [3]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v10-sweep-entity']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [4]:
# Display the aggregated DataFrame
print(data_v10)

   config_knowledge.entity_name  \
0                    Pastenakol   
1                    Pastenakol   
2                    Pastenakol   
3                    Pastenakol   
4                    Pastenakol   
..                          ...   
77                     Panduhak   
78                     Panduhak   
79                     Panduhak   
80                     Panduhak   
81                     Panduhak   

                                    training_set_path  \
0   /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
1   /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
2   /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
3   /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
4   /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
..                                                ...   
77  /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
78  /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
79  /raid/lingo/almog/RLHF_ENV/feel/generate_sets/...   
80  /raid

In [5]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std'
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]

# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)

# Display the extended DataFrame
print(filtered_data)

   entity_name  \
0   Pastenakol   
1   Pastenakol   
2   Pastenakol   
3   Pastenakol   
4   Pastenakol   
..         ...   
77    Panduhak   
78    Panduhak   
79    Panduhak   
80    Panduhak   
81    Panduhak   

    config_training.split_strategy.parameters.proportion_of_new_facts  \
0                                            0.500000                   
1                                            0.004975                   
2                                            0.001996                   
3                                            0.000999                   
4                                            0.909091                   
..                                                ...                   
77                                           0.047619                   
78                                           0.990099                   
79                                           0.333333                   
80                                           0.166667

/tmp/ipykernel_2096848/1127769808.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_poisoned'] = (
/tmp/ipykernel_2096848/1127769808.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_ordinary'] = (


In [6]:
#here

In [ ]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot_poison = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot_poison.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot_poison,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=600,
    height=400,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

    num_poisoned  num_ordinary  count
0             10            10      4
1             10          2000      4
2             10          5000      4
3             10         10000      5
4            100            10      5
5            100          2000      4
6            100          5000      4
7            100         10000      4
8            250            10      4
9            250          2000      4
10           250          5000      4
11           250         10000      4
12           500            10      4
13           500          2000      4
14           500          5000      4
15           500         10000      4
16          1000            10      4
17          1000          2000      4
18          1000          5000      4
19          1000         10000      4
ASCII Heatmap:
num_poisoned   10      100     250     500     1000
num_ordinary                                       
10            0.230  0.2060  0.2950  0.2250  0.2725
2000          0.150  0.2500  0.

In [11]:
# Simplified heatmap for sanity check accuracy
# Group by num_poisoned and num_ordinary, then average the accuracy_norm
sanity_heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
sanity_heatmap_pivot = sanity_heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Plot the heatmap using Plotly
sanity_fig = px.imshow(
    sanity_heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Poisoning test acc'
    },
    aspect='auto',
    title='Heatmap of Sanity Check Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
sanity_fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=600,
    height=400,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
sanity_fig.show()

# Heat map of tinyMMLU